# MIST · ACDC Training — RunPod
**Branch**: `baseline-try2`

**Architecture fixes applied vs. original paper repo**:
- `BottleneckBlock`: doubles channels at constant 8×8 spatial size — no `max_pool2d` (paper §3.1: *"image size remains the same"*)
- `Dilated_Conv` (SWC): dilations 2 and 3, **concatenated** then projected (paper Eq. 5-6, ablation Table 3 best config)
- Decoder block 1 output **excluded** from supervision (paper §3.3: *"we do not use the output of decoder block-1 as its image resolution is low"*) → 3 active outputs, 7 powerset subsets
- `RandomGenerator`: adds `random_zoom` and `random_shift` augmentations (paper §3.7)

**Training matches paper Section 3.7**: AdamW lr=1e-4, wd=1e-4, batch=12, img=256×256, 300 epochs, fixed LR, loss = 0.3·Dice + 0.7·CE (Eq. 13), powerset mutation.

**Run cells in order.**

In [ ]:
# Cell 2 — GPU check
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'No GPU detected!'
print('GPU check passed')

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-600:]

pkgs = [
    'scipy>=1.14.0',
    'timm==0.9.12',
    'medpy',                      # required: top-level import in utils/utils.py
    'SimpleITK',                  # required: top-level import in utils/utils.py
    'seaborn',                    # required: top-level import in utils/utils.py
    'segmentation-mask-overlay',  # required: top-level import in utils/utils.py
    'thop',                       # required: top-level import in utils/utils.py (profile, clever_format)
    'scikit-image',
    'einops',
    'tensorboardX',
    'tqdm',
    'gdown',
    'matplotlib',
    'pandas',
]
for spec in pkgs:
    code, out = pip('install', spec, '--quiet')
    status = 'OK    ' if code == 0 else 'FAILED'
    print(f'  {status}  {spec}')
    if code != 0:
        print(out)

print('\nAll dependencies installed.')

In [ ]:
# Cell 4 — Smoke test imports
import numpy as np
import scipy
import torch
import timm
import medpy.metric
import seaborn
import SimpleITK
import thop
import segmentation_mask_overlay

print(f'numpy     {np.__version__}')
print(f'scipy     {scipy.__version__}')
print(f'torch     {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'timm      {timm.__version__}')
print('medpy     OK')
print('seaborn   OK')
print('SimpleITK OK')
print('thop      OK')
print('segmentation_mask_overlay OK')
print('All imports OK')

In [ ]:
# Cell 5 — Clone / update repo  (branch: baseline-try2)
import subprocess, os, sys

REPO_URL = 'https://github.com/biancafabian/MIST.git'
BRANCH   = 'baseline-try2'
REPO_DIR = '/workspace/MIST'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo exists at {REPO_DIR}, syncing {BRANCH}...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], capture_output=True)
    r = subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'],
                       capture_output=True, text=True)
    print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
else:
    print(f'Cloning {BRANCH}...')
    r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout.strip(), r.stderr.strip())
    assert r.returncode == 0, f'git clone failed: {r.stderr}'

# Verify key files exist
for fname in ['lib/MIST.py', 'lib/networks.py', 'ACDC_train_test.py',
              'utils/utils.py', 'utils/dataset_ACDC.py']:
    path = os.path.join(REPO_DIR, fname)
    ok = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  {ok}  {fname}')

# Confirm architecture fixes are present
with open(os.path.join(REPO_DIR, 'lib', 'MIST.py')) as fh:
    mist_src = fh.read()
assert 'class BottleneckBlock' in mist_src,    'BottleneckBlock not found — push fix first'
assert 'class Bottleneck_decoder' in mist_src, 'Bottleneck_decoder not found — push fix first'
assert 'conv_d2' in mist_src,                  'Dilated_Conv fix (conv_d2) not found — push fix first'
print('BottleneckBlock         confirmed in lib/MIST.py')
print('Bottleneck_decoder      confirmed in lib/MIST.py')
print('Dilated_Conv d=2,3 cat  confirmed in lib/MIST.py')

with open(os.path.join(REPO_DIR, 'lib', 'networks.py')) as fh:
    nets_src = fh.read()
assert 'class MIST_CAM' in nets_src, 'MIST_CAM not found in lib/networks.py'
assert 'out_head1' not in nets_src,  'out_head1 still present — block-1 exclusion fix missing'
print('MIST_CAM (3 output heads) confirmed in lib/networks.py')

with open(os.path.join(REPO_DIR, 'utils', 'dataset_ACDC.py')) as fh:
    ds_src = fh.read()
assert 'def random_zoom' in ds_src,  'random_zoom not found in dataset_ACDC.py'
assert 'def random_shift' in ds_src, 'random_shift not found in dataset_ACDC.py'
print('random_zoom + random_shift  confirmed in utils/dataset_ACDC.py')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
# Cell 7 — Download raw ACDC archive from Google Drive
# Paste the file ID from your Google Drive share link:
#   https://drive.google.com/file/d/  >>>FILE_ID<<<  /view
# Supports .zip, .tar.gz, .tar

GDRIVE_FILE_ID = 'YOUR_GDRIVE_FILE_ID_HERE'   # <-- replace this

DOWNLOAD_TO = '/workspace/ACDC_raw'
import os, gdown, tarfile, zipfile

assert GDRIVE_FILE_ID != 'YOUR_GDRIVE_FILE_ID_HERE', \
    'Set GDRIVE_FILE_ID to the file ID from your Google Drive share link!'

os.makedirs(DOWNLOAD_TO, exist_ok=True)

# Download
archive_path = os.path.join(DOWNLOAD_TO, 'acdc_raw_archive')
print(f'Downloading from Google Drive ({GDRIVE_FILE_ID})...')
gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}',
               archive_path, quiet=False, fuzzy=True)

# Detect format and extract
print('Extracting...')
if tarfile.is_tarfile(archive_path):
    with tarfile.open(archive_path) as tf:
        tf.extractall(DOWNLOAD_TO)
    os.remove(archive_path)
elif zipfile.is_zipfile(archive_path):
    with zipfile.ZipFile(archive_path) as zf:
        zf.extractall(DOWNLOAD_TO)
    os.remove(archive_path)
else:
    # gdown may have already saved with the real extension — try renaming
    import subprocess
    result = subprocess.run(['file', archive_path], capture_output=True, text=True)
    print('file type:', result.stdout)
    raise RuntimeError(f'Unknown archive format. Check {archive_path} manually.')

# Show what was extracted so you can set RAW_DIR correctly in Cell 7c
print(f'\nExtracted to {DOWNLOAD_TO}:')
for entry in sorted(os.listdir(DOWNLOAD_TO))[:20]:
    full = os.path.join(DOWNLOAD_TO, entry)
    kind = 'DIR ' if os.path.isdir(full) else 'FILE'
    print(f'  {kind}  {entry}')

# Auto-detect the folder containing patient001/
import glob
candidates = glob.glob(os.path.join(DOWNLOAD_TO, '**/patient001'), recursive=True)
if candidates:
    detected = os.path.dirname(candidates[0])
    print(f'\nDetected patient folders under: {detected}')
    print(f'  -> Set RAW_DIR = {repr(detected)}  in Cell 7c')
else:
    print('\nCould not auto-detect patient001/ — check the extracted structure above')
    print('  -> Set RAW_DIR in Cell 7c to the folder containing patient001/, patient002/, ...')

## Data Preprocessing
Run **Cell 7b** if your ACDC data is already preprocessed and in `./data/ACDC/` — skip to **Cell 7b-skip**.
Run **Cell 7c** if you have the **raw** ACDC `.nii.gz` files (the original challenge download). This is required to produce 3D test volumes for proper volume-level evaluation matching the paper.

In [ ]:
# Cell 7c — Preprocess raw ACDC .nii.gz files
# Skip this cell if you already have correctly preprocessed data in ./data/ACDC/
#
# Set RAW_DIR to the folder that contains patient001/, patient002/, ..., patient100/
# (the "training" subdirectory of the ACDC challenge download).
#
# What this produces:
#   train/  — 2D slices .npz  (img: H×W, label: H×W)   patients 001-070
#   valid/  — 2D slices .npz  (img: H×W, label: H×W)   patients 071-080
#   test/   — 3D volumes .npz (img: S×H×W, label: S×H×W) patients 081-100
#   lists_ACDC/train.txt, valid.txt, test.txt
#
# Normalisation: per-volume min-max to [0, 1].
# Both ED and ES frames are included for every split.

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'nibabel', '--quiet'], check=True)
print('nibabel installed')

import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

RAW_DIR = '/workspace/ACDC_raw/training'   # <-- set this to your raw ACDC training/ folder
OUT_DIR = './data/ACDC'

assert os.path.isdir(RAW_DIR), (
    f'RAW_DIR not found: {RAW_DIR}\n'
    f'Set RAW_DIR to the folder containing patient001/, patient002/, ..., patient100/')

TRAIN_IDS = list(range(1,  71))
VALID_IDS = list(range(71, 81))
TEST_IDS  = list(range(81, 101))

for sub in ['train', 'valid', 'test', 'lists_ACDC']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)


def read_ed_es(patient_dir):
    cfg = os.path.join(patient_dir, 'Info.cfg')
    ed = es = None
    with open(cfg) as fh:
        for line in fh:
            if line.startswith('ED:'):
                ed = int(line.split(':')[1].strip())
            elif line.startswith('ES:'):
                es = int(line.split(':')[1].strip())
    assert ed is not None and es is not None, f'Cannot parse ED/ES from {cfg}'
    return ed, es


def normalize_volume(vol):
    vmin, vmax = float(vol.min()), float(vol.max())
    if vmax - vmin < 1e-8:
        return np.zeros_like(vol, dtype=np.float32)
    return ((vol - vmin) / (vmax - vmin)).astype(np.float32)


def process_patient(pid, split, train_names, valid_names, test_names):
    pname = f'patient{pid:03d}'
    pdir  = os.path.join(RAW_DIR, pname)
    if not os.path.isdir(pdir):
        print(f'  WARNING: {pdir} not found — skipping')
        return

    ed_frame, es_frame = read_ed_es(pdir)

    for frame_num in [ed_frame, es_frame]:
        fstr     = f'{frame_num:02d}'
        img_path = os.path.join(pdir, f'{pname}_frame{fstr}.nii.gz')
        lbl_path = os.path.join(pdir, f'{pname}_frame{fstr}_gt.nii.gz')

        if not os.path.exists(img_path) or not os.path.exists(lbl_path):
            print(f'  WARNING: missing {img_path} or {lbl_path} — skipping')
            continue

        # nibabel loads (H,W,S) → transpose to (S,H,W)
        img_vol = np.transpose(nib.load(img_path).get_fdata().astype(np.float32), (2, 0, 1))
        lbl_vol = np.transpose(nib.load(lbl_path).get_fdata().astype(np.uint8),   (2, 0, 1))
        img_vol = normalize_volume(img_vol)

        if split == 'test':
            fname = f'{pname}_frame{fstr}.npz'
            np.savez_compressed(os.path.join(OUT_DIR, 'test', fname),
                                img=img_vol, label=lbl_vol)
            test_names.append(fname)
        else:
            subdir = os.path.join(OUT_DIR, split)
            for si in range(img_vol.shape[0]):
                fname = f'{pname}_frame{fstr}_slice{si:03d}.npz'
                np.savez_compressed(os.path.join(subdir, fname),
                                    img=img_vol[si], label=lbl_vol[si])
                (train_names if split == 'train' else valid_names).append(fname)


train_names, valid_names, test_names = [], [], []

print('Train — patients 001-070')
for pid in tqdm(TRAIN_IDS):
    process_patient(pid, 'train', train_names, valid_names, test_names)
print(f'  {len(train_names)} slices')

print('Valid — patients 071-080')
for pid in tqdm(VALID_IDS):
    process_patient(pid, 'valid', train_names, valid_names, test_names)
print(f'  {len(valid_names)} slices')

print('Test  — patients 081-100')
for pid in tqdm(TEST_IDS):
    process_patient(pid, 'test', train_names, valid_names, test_names)
print(f'  {len(test_names)} volumes (each is a full S×H×W volume)')

lists_dir = os.path.join(OUT_DIR, 'lists_ACDC')
with open(os.path.join(lists_dir, 'train.txt'), 'w') as f:
    f.write('\n'.join(train_names) + '\n')
with open(os.path.join(lists_dir, 'valid.txt'), 'w') as f:
    f.write('\n'.join(valid_names) + '\n')
with open(os.path.join(lists_dir, 'test.txt'), 'w') as f:
    f.write('\n'.join(test_names) + '\n')

print(f'\nPreprocessing complete.')
print(f'  train.txt : {len(train_names)} entries')
print(f'  valid.txt : {len(valid_names)} entries')
print(f'  test.txt  : {len(test_names)} entries  (3D volumes)')

In [ ]:
# Cell 8 — Verify dataset
# All splits (train/valid/test) are 2D slices stored as individual .npz files.
# Keys are 'img' (H,W) and 'label' (H,W).
import os, glob
import numpy as np

TRAIN_DIR = './data/ACDC/train'
VALID_DIR = './data/ACDC/valid'
TEST_DIR  = './data/ACDC/test'

train_files = glob.glob(os.path.join(TRAIN_DIR, '*.npz'))
valid_files = glob.glob(os.path.join(VALID_DIR, '*.npz'))
test_files  = glob.glob(os.path.join(TEST_DIR,  '*.npz'))
print(f'Train slices : {len(train_files):4d}  (paper: ~1304)')
print(f'Valid slices : {len(valid_files):4d}  (paper: ~182)')
print(f'Test  slices : {len(test_files):4d}  (paper: ~370 slices from 20 cases)')

if train_files:
    d = np.load(train_files[0])
    img, lbl = d['img'], d['label']
    print(f'Train sample  img={img.shape}  label={lbl.shape}  '
          f'range=[{img.min():.3f}, {img.max():.3f}]  '
          f'classes={sorted(set(lbl.ravel().tolist()))}')

if test_files:
    d = np.load(test_files[0])
    img, lbl = d['img'], d['label']
    print(f'Test  sample  img={img.shape}  label={lbl.shape}  '
          f'classes={sorted(set(lbl.ravel().tolist()))}')
    print(f'Test ndim={img.ndim}  (2D slice-per-file, matching train/valid format)')

print('Dataset OK')

In [ ]:
# Cell 9 — Pre-download MaxViT pretrained weights
import os, torch

WEIGHTS_PATH = './pretrained_pth/maxvit/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth'
WEIGHTS_URL  = ('https://github.com/rwightman/pytorch-image-models/releases/'
                'download/v0.1-weights-maxx/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth')

if os.path.exists(WEIGHTS_PATH):
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Weights already present ({size_mb:.0f} MB): {WEIGHTS_PATH}')
else:
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    print('Downloading MaxViT weights...')
    torch.hub.download_url_to_file(WEIGHTS_URL, WEIGHTS_PATH)
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Saved ({size_mb:.0f} MB): {WEIGHTS_PATH}')
print('MaxViT weights OK')

In [ ]:
# Cell 11 — Training configuration  (paper Section 3.7)
BATCH_SIZE   = 12
LR           = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 300
IMG_SIZE     = 256
NUM_CLASSES  = 4       # background + RV + Myo + LV
SEED         = 2222

# Paper Eq.13: L_total = gamma * L_DICE + (1-gamma) * L_CE,  gamma = 0.3
LC_DICE = 0.3
LC_CE   = 0.7

# Boundary loss (Kervadec et al. 2019) - ramped in slowly.
# Has no lower bound of 0 like Dice/CE, so full weight from epoch 0
# destabilizes training - ramp linearly over the first N epochs instead.
BOUNDARY_WEIGHT_MAX  = 0.0   # disabled: two formulations tried, neither beat baseline
BOUNDARY_RAMP_EPOCHS = 50

# Per-class loss weights [background, RV, Myo, LV]. RV is consistently the
# weakest class in validation (~0.75 vs ~0.87-0.89 for Myo/LV) - upweight it
# so Dice+CE gradient targets the actual bottleneck instead of spreading evenly.
CLASS_WEIGHTS = [1.0, 1.8, 1.0, 1.0]

DATA_DIR = './data/ACDC'
LIST_DIR = './data/ACDC/lists_ACDC'
TEST_DIR = './data/ACDC/test'
SAVE_DIR = './model_pth'

print(f'batch={BATCH_SIZE}  lr={LR}  wd={WEIGHT_DECAY}  epochs={MAX_EPOCHS}')
print(f'img={IMG_SIZE}x{IMG_SIZE}  classes={NUM_CLASSES}  seed={SEED}')
print(f'loss = {LC_DICE}*Dice + {LC_CE}*CE  (powerset mutation, 3 active outputs, 7 subsets) + boundary(ramp 0->{BOUNDARY_WEIGHT_MAX} over {BOUNDARY_RAMP_EPOCHS} epochs)')

In [ ]:
# Cell 12 — Initialise model, data loaders, losses, optimiser
import os, sys, time, random
import numpy as np
import torch
import torch.optim as optim
from torch.nn.modules.loss import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom as zoom_fn

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

from lib.networks import MIST_CAM
from utils.utils import DiceLoss, BoundaryLoss, powerset
from utils.dataset_ACDC import ACDCdataset, RandomGenerator

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Snapshot directory
run_id        = time.strftime('%H%M%S')
snapshot_path = os.path.join(SAVE_DIR, f'MIST_CAM_{IMG_SIZE}_run{run_id}')
os.makedirs(snapshot_path, exist_ok=True)
print(f'Snapshot dir: {snapshot_path}')

# Model
net = MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()
n_params = sum(p.numel() for p in net.parameters()) / 1e6
print(f'Model: MIST_CAM  params={n_params:.1f}M')

# torch.compile: fuses ops for ~15-20% faster iterations (PyTorch 2.0+).
# First epoch will be slower (JIT compilation), subsequent epochs benefit.
if hasattr(torch, 'compile'):
    try:
        net = torch.compile(net, mode='reduce-overhead')
        print('torch.compile enabled')
    except Exception as e:
        print(f'torch.compile skipped: {e}')
else:
    print('torch.compile not available (PyTorch < 2.0)')

# Losses
class_weights_t = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).cuda()
ce_loss   = CrossEntropyLoss(weight=class_weights_t)
dice_loss = DiceLoss(NUM_CLASSES)
boundary_loss = BoundaryLoss(NUM_CLASSES)

# Data loaders
train_dataset = ACDCdataset(
    DATA_DIR, LIST_DIR, split='train',
    transform=transforms.Compose([RandomGenerator(output_size=[IMG_SIZE, IMG_SIZE])]))
val_dataset = ACDCdataset(DATA_DIR, LIST_DIR, split='valid')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
valloader    = DataLoader(val_dataset,   batch_size=1,          shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print(f'Train: {len(train_dataset)} slices  |  Val: {len(val_dataset)} slices')
print(f'Train iters/epoch: {len(train_loader)}')

# Optimiser — paper Section 3.7: AdamW, fixed LR (no schedule)
optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler()

# Save run config
with open(os.path.join(snapshot_path, 'config.txt'), 'w') as f:
    f.write('model=MIST_CAM\n')
    f.write('encoder=maxxvit_rmlp_small_rw_256 (pretrained ImageNet)\n')
    f.write('decoder=CAM (BottleneckBlock + Bottleneck_decoder + 3xBlock_decoder)\n')
    f.write('swc=dilations_2_3_concat\n')
    f.write('decoder_outputs=3 (block-1 excluded)\n')
    f.write(f'batch={BATCH_SIZE}\n')
    f.write(f'lr={LR}\n')
    f.write(f'weight_decay={WEIGHT_DECAY}\n')
    f.write('lr_schedule=fixed\n')
    f.write(f'epochs={MAX_EPOCHS}\n')
    f.write(f'img_size={IMG_SIZE}\n')
    f.write(f'num_classes={NUM_CLASSES}\n')
    f.write(f'seed={SEED}\n')
    f.write(f'loss={LC_DICE}*Dice+{LC_CE}*CE_powerset_mutation_7subsets\n')
    f.write('augmentation=rot_flip+rotate+zoom(0.85-1.25)+shift(10%)\n')
print('config.txt saved')

In [ ]:
# Cell 13 — Training loop
# Loss: 0.3*Dice + 0.7*CE per 7 non-empty powerset subsets of 3 active outputs (paper Eq.12-13).
# LR schedule: fixed (paper Section 3.7).
# Checkpoints: last.pth (resumable, every epoch) + best.pth (best mean per-class val Dice).
from tqdm import tqdm

# 3 outputs from decoder blocks 2, 3, 4 — block 1 excluded per paper
# powerset([0,1,2]) yields 2^3 - 1 = 7 non-empty subsets
l  = [0, 1, 2]
ss = [s for s in powerset(l)]
print(f'Active outputs: {len(l)}  |  Powerset subsets: {len(ss)}')

CLASS_NAMES = ['RV', 'Myo', 'LV']


def dice_per_class(pred, gt, c):
    """Dice for a single class c on a 2D slice."""
    p = (pred == c).astype(bool)
    g = (gt   == c).astype(bool)
    inter = np.count_nonzero(p & g)
    denom = np.count_nonzero(p) + np.count_nonzero(g)
    return 2.0 * inter / denom if denom else 0.0


def do_val():
    """Per-class Dice averaged over all validation slices — matches paper Table 1 metric."""
    net.eval()
    class_dice = np.zeros(NUM_CLASSES - 1)   # one entry per foreground class
    with torch.no_grad():
        for vb in valloader:
            img = vb['image'].squeeze(0).cpu().numpy()   # (H, W)
            lbl = vb['label'].squeeze(0).cpu().numpy()
            h, w = img.shape[0], img.shape[1]
            if h != IMG_SIZE or w != IMG_SIZE:
                img = zoom_fn(img, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().cuda()
            with autocast():
                P = net(t)
            out = sum(P)
            out = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom_fn(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            for ci, c in enumerate(range(1, NUM_CLASSES)):
                class_dice[ci] += dice_per_class(out, lbl, c)
    net.train()
    class_dice /= len(valloader)
    return class_dice          # shape (3,): [RV_dice, Myo_dice, LV_dice]


Loss, ValDice = [], []
best_dcs   = 0.80
best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
iter_num   = 0

print(f'Training {MAX_EPOCHS} epochs | {len(train_loader)} iters/epoch')
print(f'Snapshot: {snapshot_path}')

for epoch in tqdm(range(MAX_EPOCHS)):
    net.train()
    epoch_loss = 0.0
    bnd_weight = BOUNDARY_WEIGHT_MAX * min(1.0, epoch / BOUNDARY_RAMP_EPOCHS)

    for sampled_batch in train_loader:
        imgs   = sampled_batch['image'].float().cuda()
        labels = sampled_batch['label'].float().cuda()

        with autocast():
            P    = net(imgs)
            loss = 0.0
            for s in ss:
                if s == []:
                    continue
                iout      = sum(P[idx] for idx in s)
                loss_dice = dice_loss(iout, labels, softmax=True, weight=CLASS_WEIGHTS)
                loss_ce   = ce_loss(iout, labels.long())
                loss     += LC_DICE * loss_dice + LC_CE * loss_ce

            if bnd_weight > 0:
                full_pred = sum(P)
                loss = loss + bnd_weight * boundary_loss(full_pred, labels.long())

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Fixed LR — paper Section 3.7 (no schedule)
        iter_num   += 1
        epoch_loss += loss.item()
        if iter_num % 100 == 0:
            tqdm.write(f'  iter {iter_num:5d}  loss {loss.item():.4f}  lr {LR:.2e}')

    Loss.append(epoch_loss / len(train_dataset))

    # Resumable checkpoint every epoch
    torch.save({
        'epoch':                epoch,
        'iter_num':             iter_num,
        'model_state_dict':     net.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_dcs':             best_dcs,
    }, os.path.join(snapshot_path, 'last.pth'))

    per_class = do_val()          # [RV, Myo, LV]
    mean_dcs  = per_class.mean()
    ValDice.append(mean_dcs)

    tqdm.write(
        f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  '
        f'loss={Loss[-1]:.5f}  bnd_w={bnd_weight:.3f}  '
        f'val RV={per_class[0]:.3f} Myo={per_class[1]:.3f} LV={per_class[2]:.3f}  '
        f'mean={mean_dcs:.4f}  best={best_dcs:.4f}'
    )

    if mean_dcs > best_dcs:
        best_dcs   = mean_dcs
        best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        torch.save(net.state_dict(), os.path.join(snapshot_path, 'best.pth'))
        tqdm.write(f'  -> New best mean Dice: {best_dcs:.4f}  '
                   f'(RV={per_class[0]:.3f} Myo={per_class[1]:.3f} LV={per_class[2]:.3f})')

torch.save(best_state, os.path.join(snapshot_path, 'best.pth'))
print(f'Training complete.  Best val mean Dice: {best_dcs:.4f}')
print(f'Checkpoint: {snapshot_path}/best.pth')

In [ ]:
# Cell 14 — Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(Loss, color='steelblue')
axes[0].set_title('MIST_CAM — Train Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

best_epoch = ValDice.index(max(ValDice))
axes[1].plot(ValDice, color='seagreen')
axes[1].axhline(max(ValDice), color='red', linestyle='--', alpha=0.6,
                label=f'Best: {max(ValDice):.4f} @ epoch {best_epoch + 1}')
axes[1].set_title('Validation Dice (mean of RV/Myo/LV per-class Dice)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')
print(f'Best val Dice: {max(ValDice):.4f}  at epoch {best_epoch + 1}')

In [ ]:
# Cell 15 — Package checkpoint for download
import shutil, os
from IPython.display import FileLink, display

zip_base = f'/workspace/MIST_CAM_{IMG_SIZE}_run{run_id}'
print(f'Zipping {snapshot_path} ...')
shutil.make_archive(zip_base, 'zip', snapshot_path)
zip_path = zip_base + '.zip'
size_mb  = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({size_mb:.0f} MB)')
display(FileLink(zip_path))

In [ ]:
# Cell 15b — Optional: upload to Google Drive with rclone
# Configure rclone first: !rclone config
RUN_UPLOAD    = False
REMOTE        = 'gdrive'
GDRIVE_FOLDER = 'MIST_results'

if RUN_UPLOAD:
    import subprocess
    r = subprocess.run(
        ['rclone', 'copy', zip_path, f'{REMOTE}:{GDRIVE_FOLDER}/', '-v'],
        capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print('rclone FAILED:', r.stderr)
else:
    print('Upload skipped (set RUN_UPLOAD=True to enable).')

## Volume-Level Inference
Run after training. Loads `best.pth`, iterates test volumes (each `.npz` is a full 3D volume), runs per-slice inference, and reports **per-class Dice and HD95** — the metrics used in paper Table 1.

Paper target: **Mean Dice 92.56%** — RV 91.23, Myo 90.31, LV 96.14.

In [ ]:
# Cell 16 — Inference: per-class Dice + HD95 (scipy-based, no medpy)
# Test data is 2D slices stored per-file, same format as train/valid.
# Metrics are computed per slice and averaged — matching the original ACDC_train_test.py behaviour.
import os, sys, glob
import numpy as np
import torch
from tqdm import tqdm
from scipy.ndimage import zoom as zoom_fn, binary_erosion, distance_transform_edt
from utils.utils import keep_largest_component, predict_tta

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

# --- Metric helpers (scipy-based, no medpy) ---
def dice_cls(pred, gt, c):
    p = (pred == c).astype(bool)
    g = (gt   == c).astype(bool)
    inter = np.count_nonzero(p & g)
    denom = np.count_nonzero(p) + np.count_nonzero(g)
    return 2.0 * inter / denom if denom else 0.0

def hd95_cls(pred, gt, c):
    p = (pred == c).astype(bool)
    g = (gt   == c).astype(bool)
    if not p.any() and not g.any():
        return 0.0           # both empty — perfect match
    if not p.any() or not g.any():
        return np.inf        # one is empty — no overlap possible
    rb = p ^ binary_erosion(p)
    sb = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~g)[rb]
    d2 = distance_transform_edt(~p)[sb]
    return float(np.percentile(np.hstack([d1, d2]), 95))

# --- Load model ---
CHECKPOINT = os.path.join(snapshot_path, 'best.pth')
print(f'Loading checkpoint: {CHECKPOINT}')

from lib.networks import MIST_CAM as _MIST_CAM
net_inf = _MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()

state = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
# torch.compile wraps keys with _orig_mod. prefix — strip it for plain model loading
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
    print('Stripped _orig_mod. prefix from torch.compile state dict')
net_inf.load_state_dict(state)
net_inf.eval()
print('Model loaded')

# --- Load test file list ---
list_file = os.path.join(LIST_DIR, 'test.txt')
with open(list_file) as fh:
    test_names = [line.strip() for line in fh if line.strip()]
print(f'Test items: {len(test_names)}')

# --- Inference loop ---
# Each item is a 2D slice .npz (same format as train/valid).
# If a file turns out to be a 3D volume (S,H,W) the loop handles it slice-by-slice.
all_metrics = []   # list of per-item [(dice_c1, hd95_c1), (dice_c2, hd95_c2), ...]

with torch.no_grad():
    for name in tqdm(test_names, desc='Test slices'):
        filepath = os.path.join(TEST_DIR, name)
        data     = np.load(filepath)
        img_data = data['img']     # (H,W) or (S,H,W)
        lbl_data = data['label']   # (H,W) or (S,H,W)

        if img_data.ndim == 2:
            # --- 2D slice ---
            h, w = img_data.shape
            slc = img_data
            if h != IMG_SIZE or w != IMG_SIZE:
                slc = zoom_fn(slc, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(slc).unsqueeze(0).unsqueeze(0).float().cuda()
            with torch.cuda.amp.autocast():
                probs = predict_tta(net_inf, t)
            pred = probs.argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                pred = zoom_fn(pred, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            lbl = lbl_data
        else:
            # --- 3D volume (S,H,W) ---
            pred = np.zeros_like(lbl_data)
            for si in range(img_data.shape[0]):
                slc  = img_data[si]
                h, w = slc.shape
                if h != IMG_SIZE or w != IMG_SIZE:
                    slc = zoom_fn(slc, (IMG_SIZE / h, IMG_SIZE / w), order=3)
                t = torch.from_numpy(slc).unsqueeze(0).unsqueeze(0).float().cuda()
                with torch.cuda.amp.autocast():
                    probs = predict_tta(net_inf, t)
                out = probs.argmax(dim=1).squeeze(0).cpu().numpy()
                if h != IMG_SIZE or w != IMG_SIZE:
                    out = zoom_fn(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
                pred[si] = out
            pred = keep_largest_component(pred, NUM_CLASSES)  # drop spurious islands, test-time only
            lbl = lbl_data

        item_metrics = []
        for c in range(1, NUM_CLASSES):   # skip background (class 0)
            item_metrics.append((dice_cls(pred, lbl, c),
                                 hd95_cls(pred, lbl, c)))
        all_metrics.append(item_metrics)

# --- Report ---
M           = np.array([[m[c] for c in range(NUM_CLASSES - 1)] for m in all_metrics])
# M shape: (N, n_classes-1, 2)  where [:,:,0]=dice  [:,:,1]=hd95
class_names = ['RV', 'Myo', 'LV']

print(f'\n=== Test Results  ({len(all_metrics)} items) ===')
for ci, cname in enumerate(class_names):
    dc_vals  = M[:, ci, 0]
    hd_vals  = M[:, ci, 1]
    finite   = np.isfinite(hd_vals)
    dc_mean  = dc_vals.mean() * 100
    hd_mean  = hd_vals[finite].mean() if finite.any() else float('inf')
    print(f'  Class {ci+1} ({cname:<3})  Dice = {dc_mean:.2f}%   HD95 = {hd_mean:.2f}')

mean_dice = M[:, :, 0].mean() * 100
print(f'  ----------------------------------------')
print(f'  Mean Dice  : {mean_dice:.2f}%  (paper target: 92.56%)')